## 06 - Lines and Distances

### Goal

- Last lesson we loaded the WDO package and used some of the given spatial functions. Now we are going to put package to use and calculate many distances and draw some lines. 
- Using any file that has "world cities" in the name:
  
  - [WorldCitiesGeo](../data/WorldCitiesGeo/) (and all its geojson files in the directory)
  - [world_cities_fixed.json](../data/world_cities_fixed.json)
  - [world_cities_by_time-zone.json](../data/world_cities_by_time-zone.json)
  
- These contain ALL THE CITIES IN THE WORLD! Well ... most of the cities.  You know like the ones that ... have people.  I mean `Burkburnett` probably isn't in the file, but it does have people, so ... (update ... I just checked, and Burk is in the file!)
  
- Keep reading


### Tasks

- Read in a file containing the locations to cities all over the world ([world_cities_large.json](./../data/world_cities_large.json)).
- We would like to draw a line from MSU (or close to it) to each city in the file, but thats not feasible as the file is too large. 
- Looking at the options (files) above, you should have plenty of choices for filtering down to a manageable size of cities to draw lines to. 


## Possible Bonus

- Randomly choose cities from the file, and stop choosing when the distance goes over some threshold (e.g 10000km).

In [ ]:
from pathlib import Path
import json
import sys

import folium


cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()), None)
assert repo_root is not None, "❌ Could not locate the project root."

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from wdo.geo import haversine_km

data_dir = repo_root / "data"
world_geo_dir = data_dir / "WorldCitiesGeo"

selected_files = [
    world_geo_dir / "America-New_York.geojson",
    world_geo_dir / "Europe-London.geojson",
    world_geo_dir / "Asia-Tokyo.geojson",
]

for target in selected_files:
    assert target.exists(), f"❌ Missing city file: {target}"

msu = {"name": "MSU Texas", "lat": 33.87372, "lon": -98.51947}


**FYI:**

These previous three lessons will be helpful: 
- [03-Style_W_Logic](./03-Style_W_Logic.ipynb)
- [04-Distance](./04-Distance.ipynb)
- [05-Geo_and_Json_overview](./05-Geo_and_Json_overview.ipynb)



In [ ]:
city_records = []

for geojson_file in selected_files:
    data = json.loads(geojson_file.read_text())
    timezone_name = geojson_file.stem

    for feature in data["features"][:4]:
        lon, lat = feature["geometry"]["coordinates"]
        city_name = feature["properties"].get("name", "Unknown")
        distance_km = haversine_km(msu["lat"], msu["lon"], lat, lon)

        city_records.append(
            {
                "name": city_name,
                "timezone": timezone_name,
                "lat": lat,
                "lon": lon,
                "distance_km": distance_km,
            }
        )

city_records = sorted(city_records, key=lambda item: item["distance_km"])


def distance_color(distance_km):
    if distance_km < 2500:
        return "green"
    if distance_km < 8000:
        return "orange"
    return "red"


m = folium.Map(location=[msu["lat"], msu["lon"]], zoom_start=2, tiles="CartoDB positron")

folium.Marker(
    [msu["lat"], msu["lon"]],
    tooltip=msu["name"],
    icon=folium.Icon(color="blue", icon="home", prefix="fa"),
).add_to(m)

for city in city_records:
    color = distance_color(city["distance_km"])
    folium.PolyLine(
        locations=[[msu["lat"], msu["lon"]], [city["lat"], city["lon"]]],
        color=color,
        weight=2,
        opacity=0.65,
        tooltip=f"{city['name']} ({city['timezone']})",
    ).add_to(m)
    folium.CircleMarker(
        location=[city["lat"], city["lon"]],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{city['name']}: {city['distance_km']:.1f} km",
    ).add_to(m)

print(f"Prepared {len(city_records)} routes from {msu['name']}.")
for city in city_records[:5]:
    print(f"{city['name']:20} {city['timezone']:18} {city['distance_km']:8.1f} km")

m


This lesson helps with **📏 Milestone 2** 